# grad-expressed-in-out — worked example 3: log1p backward expressed via cached output

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-expressed-in-out`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For `out = log(1 + x)`, the derivative is `1 / (1 + x)`. While we could recompute `1 + x`, we can instead recover it as `exp(out)` since `out = log(1+x)` implies `exp(out) = 1+x`. The backward is therefore `grad_out / exp(out)` or equivalently `grad_out * exp(-out)`. This avoids storing `x` across the forward-backward boundary while still computing the correct local gradient.

## Worked solution

**Step 1 — derive the out-form.** `out = log(1+x)`, so `exp(out) = 1+x`. The derivative `d/dx log(1+x) = 1/(1+x) = exp(-out)`.

**Step 2 — implement.** `dL/dx = grad_out * exp(-out)`. Notice we only need `out` — no `x` needed at all.

**Step 3 — verify against the x-based form.** `1 / (1 + x)` should equal `exp(-out)` up to floating-point precision. We test this explicitly.

**Step 4 — check x-independence.** Deliberately pass a wrong `x` value — the result should not change, confirming that `x` goes unused in the implementation.

In [ ]:
import torch as t

t.manual_seed(2)

def log1p_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    # out = log(1+x)  =>  exp(-out) = 1/(1+x) = d/dx log(1+x)
    return grad_out * t.exp(-out)

# Exercise
t.manual_seed(2)
x = t.abs(t.randn(5)) + 0.1  # positive values to keep log1p well-defined
out = t.log1p(x)
grad_out = t.ones_like(x)

result = log1p_back(grad_out, out, x)

# Verify: 1/(1+x) should match exp(-out)
expected = grad_out / (1.0 + x)
assert t.allclose(result, expected, atol=1e-6), f"Expected {expected}, got {result}"
print(f"x:          {x.tolist()}")
print(f"out=log1p:  {out.tolist()}")
print(f"dL/dx:      {result.tolist()}")

# x-independence check
wrong_x = t.randn_like(x) * 100
result_wrong_x = log1p_back(grad_out, out, wrong_x)
assert t.allclose(result_wrong_x, result), "x is supposed to be unused"
print("Confirmed: log1p_back uses only out, not x.")